In [1]:
# Add autoreload at the top of the notebook you're working on 
# in order for it to auto refresh when you change the 'project_package'
%load_ext autoreload
%autoreload 2

# Initialize dummy dataset

In [2]:
import pandas as pd
import numpy as np

from project_package.data_preprocessing.utils import create_user_preference

In [3]:
movie_metadata = pd.read_csv(
    '../data/external/movielense_25m/movies_metadata.csv',
    usecols = ['id','original_title','overview','genres','original_language',
               'adult','runtime','revenue','vote_average','vote_count']
    )
movie_metadata['genres'] = np.array(movie_metadata['genres'].apply(lambda x:[item['name'] for item in eval(x)]))

movie_metadata['id'] = pd.to_numeric(movie_metadata['id'], errors='coerce')
movie_metadata.dropna(subset=['id'],inplace=True)
movie_metadata['id'] = movie_metadata['id'].astype(int)
movie_metadata.rename(columns={'id':'movieId'},inplace=True)

movie_metadata = movie_metadata.drop_duplicates('movieId').reset_index(drop=True)

movie_metadata['genres'] = movie_metadata['genres'].apply(
    lambda x: ['Unknown'] if isinstance(x, list) and len(x) == 0 else x
)

movie_metadata.head(3)

,adult,genres,movieId,original_language,original_title,overview,revenue,runtime,vote_average,vote_count
0,False,"[Animation, Comedy, Family]",862,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",373554033.0,81.0,7.7,5415.0
1,False,"[Adventure, Fantasy, Family]",8844,en,Jumanji,When siblings Judy and Peter discover an encha...,262797249.0,104.0,6.9,2413.0
2,False,"[Romance, Comedy]",15602,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,0.0,101.0,6.5,92.0


In [4]:
# should sort reviews by user first
user_reviews = pd.read_csv(
    '../data/external/movielense_25m/ratings_small.csv',
    usecols = ['userId','movieId','rating'],
    )
user_reviews.head(5)

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


In [5]:
criteria_dict = dict(
    genres = (0.3,'multiple'),  # set threshold to determine a user preferred labels and the label type (multiple labels in a cell or just 1)
    original_language = (0.1,'single')
)

preference_df = create_user_preference(
    movie_metadata,'movieId',
    user_reviews,'userId',
    criteria_dict,
    'rating',
    user_batch=200
)

preference_df.head(3)

Processing batches:   0%|          | 0/4 [00:00<?, ?it/s]

,userId,genres,original_language
0,1,[Comedy],[en]
1,2,[Drama],[en]
2,3,[Drama],"[en, fr]"
